# GV Exercise – Modular Pipeline Notebook

This notebook demonstrates the exact same data preparation, training, and inference flow as the production code by importing the reusable modules under `src/`.

It intentionally avoids ad-hoc pandas logic so experiments stay aligned with the CLI scripts and MLflow-tracked runs.

In [1]:
%pip install seaborn
%pip install optuna

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## Phase 1: Environment Setup

Bootstrap project imports and configuration. This sets up the shared imports, config, and logging so experiments match CLI runs.

In [2]:
import sys
import json
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import xgboost as xgb

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.common import load_config
from src.common.logging import init_logging
from src.data import build_company_founded_lookup, clean, load_raw, load_targets
from src.features import get_pipeline
from src.models.metrics import calculate_ndcg, calculate_relevance_grade
from src.models.io import load_model_bundle
from src.predict.inference import predict_batch
from src.models.training import train_ranker

cfg = load_config()
init_logging(cfg.logging)
cfg

Config(project=ProjectConfig(root=PosixPath('/Users/yugu/Desktop/gehirn/gv_case_study/xgboost')), data=DataConfig(raw_dir=PosixPath('/Users/yugu/Desktop/gehirn/gv_case_study/xgboost/src/data/real'), curated_dir=PosixPath('/Users/yugu/Desktop/gehirn/gv_case_study/xgboost/src/data/training'), training_file=PosixPath('/Users/yugu/Desktop/gehirn/gv_case_study/xgboost/src/data/training/target_variable_training.csv'), schema_version='v1', ranking_files=RankingFiles(founder_experience='founder_experience.csv', education='education.csv', company_info='company_info.csv'), training_files=TrainingFiles(founder_experience='founder_experience_training.csv', education='education_training.csv', company_info='company_info_training.csv')), features=FeatureConfig(registry=PosixPath('/Users/yugu/Desktop/gehirn/gv_case_study/xgboost/src/data/features'), current_year=2025, numerical_columns=['experience_span_years', 'average_role_duration_years', 'education_tier', 'education_level_score', 'direct_network_s

## Phase 2: Data Loading & Feature Engineering

Build the training dataset directly from packaged loaders. This pulls packaged training data and cleans it with the same helpers used in production. It also performs manual feature engineering and builds the feature matrix.

### 2.1 Data Loading

Load the raw training data and target labels from the configured data directories.

In [3]:
raw_training = load_raw(cfg.data, dataset="training")
target_df = load_targets(cfg.data)

### 2.2 Data Cleaning & Target Engineering

Clean the raw data using standard helpers and derive the discrete relevance grade from the multiple-on-invested-capital (MOIC) target.

In [4]:
clean_training = clean(raw_training)
target_df["relevance_grade"] = target_df["multiple"].apply(calculate_relevance_grade)

### 2.3 Outcome Lookup Creation

Build lookup dictionaries for company outcomes and founded years, which are needed for feature engineering (e.g., calculating past success rates).

In [5]:
outcome_lookup = (
    target_df.set_index("company_id")["multiple"].dropna().astype(float).to_dict()
)
company_founded_lookup = build_company_founded_lookup(target_df)

### 2.4 Feature Pipeline Initialization

Initialize the feature pipeline with the configuration and the created lookups.

In [6]:
fr_pipeline = get_pipeline(cfg.features, outcome_lookup=outcome_lookup, company_founded_lookup=company_founded_lookup)

### 2.5 Feature Matrix Construction

Build the feature matrix from the clean training data using the initialized pipeline.

In [7]:
# Build the feature matrix via the feature registry to stay consistent
train_matrix = fr_pipeline.build_matrix(clean_training)

### 2.6 Label Merging

Merge the generated features with the target labels and drop any records with missing targets.

In [8]:
# Join labels back to features and drop incomplete targets
training_df = (
    train_matrix.frame
    .merge(target_df[["company_id", "relevance_grade"]], on="company_id", how="inner")
    .dropna(subset=["relevance_grade"])
    .reset_index(drop=True)
)

# Rename founded_year to company_founded for the splitter
training_df = training_df.rename(columns={"founded_year": "company_founded"})

### 2.7 Manual Feature Engineering

Apply any manual feature engineering steps, such as handling missing values or creating custom flags, and define the final list of feature columns.

In [9]:
## manual feature engineering
'''
missing_performance 1 if missing performance, 0 otherwise
'''
def manual_feature_engineering(training_df):
    training_df["missing_performance"] = training_df["performance"].apply(lambda x: 1 if pd.isna(x) else 0)
    training_df["imputed_performance"] = fr_pipeline.impute_field(training_df, "performance", method="median")
    return training_df

training_df = manual_feature_engineering(training_df)

# Keep track of the active feature list for downstream splits
feature_columns = cfg.features.numerical_columns + cfg.features.categorical_columns + [
    # "missing_performance",
    "imputed_performance",
    # "performance",
    # "exec_role_5yr",
    # "tiered_exit_1x",
    # "tiered_exit_3x",
    # "tiered_exit_10x",
    # "tiered_exit_50x",
    # "tiered_exit_100x"
]

## Phase 3: Data Splitting

Perform per-industry cohort splitting with temporal ordering. We split the data by industry and use temporal ordering (older companies in train, newer in test) to simulate a realistic production scenario.

In [10]:
from src.data.splitters import split_by_industry, save_split_summary

# Split by industry with temporal ordering (older companies in train, newer in test)
X_train, y_train, X_test, y_test, train_df, test_df, train_groups, test_groups, industry_names = split_by_industry(
    df=training_df,
    feature_cols=feature_columns,
    target_col="relevance_grade",
    train_ratio=0.8
)

# Create artifacts directory for phase 5
artifacts_dir = cfg.project.root / "artifacts" / "phase5"
artifacts_dir.mkdir(parents=True, exist_ok=True)

# Save split summary for MLflow
summary_path = artifacts_dir / "industry_split_summary.csv"
save_split_summary(train_df, test_df, train_groups, test_groups, str(summary_path))

print(f"\nTraining rows: {len(training_df)} | Feature columns: {len(feature_columns)}")
training_df.head()


=== Split Summary ===
Total samples: 4811
Train samples: 3847 (80.0%)
Test samples: 964 (20.0%)
Number of industry cohorts: 3

Train groups (per industry): [833, 2366, 648]
Test groups (per industry): [209, 592, 163]

=== Per-Industry Breakdown ===

Healthcare and Biotech:
  Total: 1042 | Train: 833 (79.9%) | Test: 209 (20.1%)
  Train labels: {0: 377, 1: 236, 2: 117, 3: 78, 4: 18, 6: 7}
  Test labels: {0: 178, 1: 23, 2: 4, 3: 4}

Information Technology:
  Total: 2958 | Train: 2366 (80.0%) | Test: 592 (20.0%)
  Train labels: {0: 1350, 1: 411, 2: 300, 3: 207, 4: 76, 5: 12, 6: 10}
  Test labels: {0: 550, 1: 21, 2: 12, 3: 2, 4: 4, 5: 3}

Other:
  Total: 811 | Train: 648 (79.9%) | Test: 163 (20.1%)
  Train labels: {0: 443, 1: 64, 2: 47, 3: 29, 4: 16, 5: 41, 6: 5, 7: 3}
  Test labels: {0: 160, 2: 3}

Split summary saved to: /Users/yugu/Desktop/gehirn/gv_case_study/xgboost/artifacts/phase5/industry_split_summary.csv

Training rows: 4811 | Feature columns: 8


,person_id,company_id,experience_span_years,average_role_duration_years,education_tier,education_level_score,direct_network_size,network_quality,team_size,industry,...,indirect_network_size,tiered_exit_1x,tiered_exit_3x,tiered_exit_10x,tiered_exit_50x,tiered_exit_100x,relevance_grade,missing_performance,performance_median,imputed_performance
0,SAZwAA4FHAkwCwkOFTQdDwwAGg==,SBVwFg0CFhgOFA==,9.752225,2.140087,4.0,3.0,14.0,2.928571,5.0,Information Technology,...,2916.0,0,0,0,0,0,0,0,0.203511,0.203511
1,SAZwAA4FHDcFDAgIAyw=,SBVwGgQTCAc=,5.166324,1.790554,4.0,3.0,3.0,2.000000,2.0,Information Technology,...,2916.0,0,0,0,0,0,3,0,10.000000,10.000000
2,SAZwAA4FHDcJCBcJFzEQBBg6WF9c,SBVwGgQfFgsOHQA=,7.000684,6.915811,5.0,1.0,14.0,3.642857,4.0,Information Technology,...,2916.0,0,0,0,0,0,4,0,10.000000,10.000000
3,SAZwAA4FHDcJDBAUAjYaCA==,SBVwGAgMDRwCBAsCBAAHGBgRDQIe,28.418891,6.978782,4.0,3.0,61.0,2.934426,3.0,Information Technology,...,2916.0,0,0,0,0,0,0,0,0.754910,0.754910
4,SAZwAA4FHDcMFwQVGDYf,SBVwHQ0HEAUGAwQ=,16.418891,5.916496,5.0,4.0,9.0,3.000000,9.0,Healthcare and Biotech,...,2916.0,0,0,0,0,0,3,0,10.000000,10.000000


In [ ]:
import optuna
import mlflow
import xgboost as xgb
import pandas as pd
from sklearn.metrics import ndcg_score
import numpy as np

# --- AutoML Configuration ---
N_TRIALS = 30  # Number of optimization trials
metric_name = "ndcg@20"

@mlflow.trace
def objective(trial):
    """
    Optuna objective function for XGBoost Ranking.
    Optimizes both hyperparameters and feature selection.
    """
    
    # --- 1. Feature Selection ---
    # Suggest which feature groups to include/exclude
    # (Assuming X_train has columns like 'education_...', 'network_...', etc.)
    
    # Get all available columns
    all_features = X_train.columns.tolist()
    
    # Example: Let Optuna decide whether to drop specific feature groups
    # You can customize this logic based on your actual feature names
    drop_education = trial.suggest_categorical("drop_education", [True, False])
    drop_network = trial.suggest_categorical("drop_network", [True, False])
    
    selected_features = all_features.copy()
    
    if drop_education:
        selected_features = [f for f in selected_features if "education" not in f]
    
    if drop_network:
        selected_features = [f for f in selected_features if "network" not in f]
        
    # Ensure we have at least one feature
    if not selected_features:
        selected_features = all_features # Fallback
        
    # Create views with selected features
    X_train_opt = X_train[selected_features]
    X_test_opt = X_test[selected_features]
    
    # --- 2. Hyperparameter Tuning ---
    params = {
        "objective": "rank:ndcg",
        "eval_metric": metric_name,
        "tree_method": "hist",
        "booster": "gbtree",
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 1.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 1.0, log=True),
    }
    
    # --- 3. Training ---
    # We use the same group structure as the main pipeline
    
    model = xgb.XGBRanker(**params)
    
    # Suppress verbose output during optimization
    model.fit(
        X_train_opt,
        y_train,
        group=train_groups,
        eval_set=[(X_test_opt, y_test)],
        eval_group=[test_groups],
        verbose=False
    )
    
    # --- 4. Evaluation ---
    # Calculate NDCG on the test set
    # Note: XGBRanker's score is usually the last metric in eval_results, 
    # but here we can just use the built-in evaluation or recalculate manually for safety.
    
    # Let's use the model's internal evaluation result on the validation set
    # (XGBoost stores evaluation results in .evals_result() if enabled, but here we just want the score)
    
    # Alternative: Predict and calculate manually to be sure
    test_scores = model.predict(X_test_opt)
    
    # Calculate mean NDCG across queries (groups)
    # We need to reconstruct the query structure for manual calculation
    # But simpler is to trust the eval_metric if we can extract it.
    # However, extracting it from the model object can be tricky without a callback.
    
    # Let's do a manual calculation using the same logic as src.models.metrics
    # Re-implementing simple group-based NDCG loop here for self-containment
    
    scores = []
    start_idx = 0
    for group_size in test_groups:
        end_idx = start_idx + group_size
        y_true_g = y_test.iloc[start_idx:end_idx].tolist()
        y_score_g = test_scores[start_idx:end_idx].tolist()
        
        # ndcg_score expects 2D arrays [n_samples, n_classes] or [n_samples, list_length]
        # Here it's [1, group_size]
        if len(set(y_true_g)) > 1: # Only calculate if there's variance
             scores.append(ndcg_score([y_true_g], [y_score_g], k=20))
        else:
             scores.append(0.0) # Or 1.0? Usually 0 if no relevant docs
             
        start_idx = end_idx
        
    mean_ndcg = np.mean(scores) if scores else 0.0
    
    # Report intermediate objective value to MLflow (optional)
    # trial.report(mean_ndcg, step=...) 
    
    return mean_ndcg

# --- Run Optimization ---
print(f"Starting AutoML optimization with {N_TRIALS} trials...")

# Create a new MLflow experiment for HPO to keep things clean
mlflow.set_tracking_uri(cfg.tracking.uri)
mlflow.xgboost.autolog()
mlflow.set_experiment("automl_optimization")

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=N_TRIALS)

print("\n=== AutoML Results ===")
print("Best Value (NDCG):", study.best_value)
print("Best Params:", study.best_params)

# --- Train Final Model with Best Params ---
print("\nTraining final model with best parameters...")

best_params = study.best_params.copy()

# Extract feature selection choices
drop_education = best_params.pop("drop_education", False)
drop_network = best_params.pop("drop_network", False)

final_features = X_train.columns.tolist()
if drop_education:
    final_features = [f for f in final_features if "education" not in f]
if drop_network:
    final_features = [f for f in final_features if "network" not in f]

print(f"Selected Features ({len(final_features)}): {final_features}")

# Update params for final training
final_xgb_params = {k: v for k, v in best_params.items() if k not in ["drop_education", "drop_network"]}
final_xgb_params["objective"] = "rank:ndcg"
final_xgb_params["tree_method"] = "hist"
final_xgb_params["eval_metric"] = metric_name

# Train final model
final_model = xgb.XGBRanker(**final_xgb_params)
final_model.fit(
    X_train[final_features],
    y_train,
    group=train_groups,
    eval_set=[(X_test[final_features], y_test)],
    eval_group=[test_groups],
    verbose=True
)

print("Final model training complete.")


/Users/yugu/Desktop/gehirn/gv_case_study/.venv/lib/python3.13/site-packages/mlflow/tracking/_tracking_service/utils.py:140: FutureWarning: Filesystem tracking backend (e.g., './mlruns') is deprecated. Please switch to a database backend (e.g., 'sqlite:///mlflow.db'). For feedback, see: https://github.com/mlflow/mlflow/issues/18534
  return FileStore(store_uri, store_uri)
[I 2025-11-22 17:22:22,244] A new study created in memory with name: no-name-e52020dc-6ba1-4edb-8553-ac17fb1bb2e2


Starting AutoML optimization with 20 trials...


[I 2025-11-22 17:22:23,015] Trial 0 finished with value: 0.39097391515225083 and parameters: {'drop_education': True, 'drop_network': True, 'n_estimators': 378, 'learning_rate': 0.061673689431548176, 'max_depth': 10, 'subsample': 0.6641623098443195, 'colsample_bytree': 0.6754861881544965, 'min_child_weight': 4, 'reg_alpha': 3.784650165933162e-07, 'reg_lambda': 4.223106633477069e-05}. Best is trial 0 with value: 0.39097391515225083.
[I 2025-11-22 17:22:23,750] Trial 1 finished with value: 0.42310786714457743 and parameters: {'drop_education': True, 'drop_network': False, 'n_estimators': 336, 'learning_rate': 0.017168291198271693, 'max_depth': 3, 'subsample': 0.6014347250288532, 'colsample_bytree': 0.720181335195167, 'min_child_weight': 4, 'reg_alpha': 1.4900819774749689e-08, 'reg_lambda': 5.423342993873569e-07}. Best is trial 1 with value: 0.42310786714457743.
[I 2025-11-22 17:22:24,431] Trial 2 finished with value: 0.06937348879520765 and parameters: {'drop_education': False, 'drop_net


=== AutoML Results ===
Best Value (NDCG): 0.6789262645864905
Best Params: {'drop_education': False, 'drop_network': False, 'n_estimators': 186, 'learning_rate': 0.017137879766579295, 'max_depth': 8, 'subsample': 0.9183197163788888, 'colsample_bytree': 0.5963535210427195, 'min_child_weight': 3, 'reg_alpha': 0.0003010097765462674, 'reg_lambda': 1.2487850735051343e-08}

Training final model with best parameters...
Selected Features (8): ['experience_span_years', 'average_role_duration_years', 'education_tier', 'education_level_score', 'direct_network_size', 'network_quality', 'team_size', 'imputed_performance']
[0]	validation_0-ndcg@20:0.17067
[1]	validation_0-ndcg@20:0.21167
[2]	validation_0-ndcg@20:0.18460
[3]	validation_0-ndcg@20:0.35270
[4]	validation_0-ndcg@20:0.47379
[5]	validation_0-ndcg@20:0.50066
[6]	validation_0-ndcg@20:0.41068
[7]	validation_0-ndcg@20:0.38859
[8]	validation_0-ndcg@20:0.49585
[9]	validation_0-ndcg@20:0.54215
[10]	validation_0-ndcg@20:0.57735
[11]	validation_0-n